# 🗡️ Mara Stone — The Immortal Assassin
## Voice Aging MVP

**Character Arc:** Mara Stone — trained from childhood as a weapon, spends her life slowly rediscovering her humanity.

**Personality Core:** a precise, controlled female assassin — economy of words, nothing wasted, flat delivery that occasionally breaks to reveal something raw underneath; voice of someone who learned not to feel and spent decades unlearning that lesson

In [ ]:
!pip install -q qwen-tts soundfile accelerate transformers

In [ ]:
import os
import gc
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
personality_core = "a precise, controlled female assassin — economy of words, nothing wasted, flat delivery that occasionally breaks to reveal something raw underneath; voice of someone who learned not to feel and spent decades unlearning that lesson"

character_data = [
    {
        "stage": "youth",
        "age": 17,
        "title": "The Weapon",
        "modifier": "flat and trained, emotion entirely suppressed, child's voice carrying an adult's controlled emptiness, eerily calm for someone so young",
        "lines": [
            "I complete the mission. That is the totality of my function. There is nothing else.",
            "Pain is information. I have learned to file it away and process it later. Or not at all.",
            "I don't have a name. I have a designation. They told me this was the same thing. I believed them for a long time."
        ]
    },
    {
        "stage": "prime",
        "age": 31,
        "title": "The Questioning",
        "modifier": "still controlled and precise, but cracks beginning — moments where genuine emotion surfaces before being locked down again, uneasy with herself",
        "lines": [
            "I have been told my entire life that I don't feel things. I'm starting to suspect that was a lie they told me for their convenience.",
            "There's a difference between completing a mission and believing in it. I've been confusing these things for years.",
            "Someone asked me yesterday what I wanted. Not what I was assigned. What I wanted. I didn't have an answer. I've been trying to find one."
        ]
    },
    {
        "stage": "middle",
        "age": 46,
        "title": "The Reclaiming",
        "modifier": "warmer but still careful, someone actively practicing emotion like a language they learned late, genuine but deliberate, the flat precision softening into something human",
        "lines": [
            "I spent seventeen years being a weapon. I've spent the last fifteen trying to remember I'm a person. It's slower work.",
            "I'm told I'm intimidating. I don't try to be. I just still haven't fully learned the parts of being human that make people comfortable.",
            "I care about things now. Real things. People. It is genuinely terrifying and I would not go back."
        ]
    },
    {
        "stage": "elder",
        "age": 63,
        "title": "The Human",
        "modifier": "quietly warm, the precision still there but serving warmth now rather than suppression, someone who found themselves and kept them, occasional wry humor from a dark past",
        "lines": [
            "I've been a weapon and a ghost and a fugitive and a protector. Human is taking the longest. I think that's right.",
            "People who knew me young are always surprised I'm still alive. I'm always surprised I'm still — me. I find I prefer the latter surprise.",
            "I don't regret any of it. That's not the same as being glad it happened. Both things can be true."
        ]
    }
]

In [ ]:
print("Loading Qwen3-TTS VoiceDesign model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
model = Qwen3TTSModel.from_pretrained(model_id, device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")
print("Model loaded successfully!")

In [ ]:
all_audio_data = []
sr = None
char_id = "mara"

for stage_info in character_data:
    stage = stage_info["stage"]
    title = stage_info["title"]
    instruct = f"{personality_core}, {stage_info['modifier']}"
    print(f"\n--- Stage: {title} ({stage_info['age']}) ---")
    print(f"Instruction: {instruct}\n")
    
    stage_audio = []
    for i, line in enumerate(stage_info["lines"]):
        print(f"Generating Line {i+1}...")
        wavs, cur_sr = model.generate_voice_design(line, language="English", instruct=instruct)
        wav_data = wavs[0]
        if sr is None:
            sr = cur_sr
        
        filename = f"{char_id}_{stage}_{i+1}.wav"
        sf.write(filename, wav_data, sr)
        
        print(f"Line {i+1}: \"{line}\"")
        display(Audio(filename))
        
        stage_audio.append(wav_data)
        
    all_audio_data.append(stage_audio[0])
    
    # Cleanup VRAM
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Life story montage: line 1 from each stage concatenated with 2.5s silence
print("\n--- Life Story Montage ---")
silence_duration = 2.5
if sr is not None:
    silence_samples = int(sr * silence_duration)
    silence = np.zeros(silence_samples, dtype=np.float32)
    
    montage = []
    for i, audio in enumerate(all_audio_data):
        montage.append(audio)
        if i < len(all_audio_data) - 1:
            montage.append(silence)
            
    montage_audio = np.concatenate(montage)
    montage_filename = f"{char_id}_montage.wav"
    sf.write(montage_filename, montage_audio, sr)
    print(f"Generated: {montage_filename}")
    display(Audio(montage_filename))
else:
    print("No audio was generated.")

In [ ]:
print("\n--- Emotional Range Test ---")
test_line = "I didn't expect to find you here. Usually people run."

emotions = [
    ("Cold", "completely dead and emotionless, flat robotic delivery, terrifyingly calm"),
    ("Amused", "dark amusement, a wry and chilling smile in the voice, slightly breathy"),
    ("Vulnerable", "genuinely surprised and slightly vulnerable, the mask slipping just for a moment, hesitant")
]

for emotion_name, emotion_mod in emotions:
    print(f"\nEmotion: {emotion_name}")
    instruct = f"{personality_core}, {emotion_mod}"
    print(f"Instruction: {instruct}")
    
    wavs, cur_sr = model.generate_voice_design(test_line, language="English", instruct=instruct)
    
    filename = f"{char_id}_emotion_{emotion_name.lower()}.wav"
    sf.write(filename, wavs[0], cur_sr)
    print(f"Line: \"{test_line}\"")
    display(Audio(filename))
    
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
!zip -q mara_stone_outputs.zip mara_*.wav
from google.colab import files
files.download('mara_stone_outputs.zip')
print("Downloaded outputs zip.")